In [ ]:
mode = 'notebook'

In [ ]:
import sys
import os
import time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pickle as pck
import gudhi as gd
from gudhi.clustering.tomato import Tomato
import multipers as mp
from multipers.filtrations.density import KDE

from tomatomp import Tomatomp

from sklearn.datasets import make_moons, load_sample_image
from sklearn.metrics import pairwise_distances, adjusted_mutual_info_score, adjusted_rand_score
from sklearn.neighbors import radius_neighbors_graph, kneighbors_graph, NearestNeighbors, KernelDensity
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, SpectralClustering, AgglomerativeClustering

from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import eigsh

import scanpy as sc
import meshplot as mp

from utils import smoothed_expression, rank_genes_tomato, rank_pair_genes_tomato, rank_genes_tomatomp_radius, rank_genes_tomatomp_outlier, rank_genes_hierarchical 
from utils import mma_score, rank_pair_genes_tomatomp_radius, rank_pair_genes_tomatomp_outlier, rank_pair_genes_hierarchical, read_off, hks 
from utils import correlation_1g, correlation_2g, top_hits_10_1g, top_hits_10_2g

In [ ]:
%matplotlib widget
%load_ext autoreload
%autoreload 2

In [ ]:
if mode == 'notebook':
    data_path_prefix = './datasets/'
else:
    data_path_prefix = './datasets/'

In [ ]:
dataset = 'synthetic' if mode == 'notebook' else sys.argv[1]
experiment = 'outliers' if mode == 'notebook' else sys.argv[2]
filename = 'toy_example_w_density.txt' if mode == 'notebook' else sys.argv[3]
#filename = 'mitosis_mod.tif' if mode == 'notebook' else sys.argv[3]
#filename = '4.off' if mode == 'notebook' else sys.argv[3]
#filename = 'kpmp_30-10125_spatial_expression.csv' if mode == 'notebook' else sys.argv[3]
#filename = 'spatial_coords.npy' if mode == 'notebook' else sys.argv[3]
subsample = 10 if mode == 'notebook' else int(sys.argv[4])

In [ ]:
quant_min_dist = 0.01 if mode == 'notebook' else float(sys.argv[5])
quant_max_dist = 0.05 if mode == 'notebook' else float(sys.argv[6])
frac_outliers = 0.01 if mode == 'notebook' else float(sys.argv[7])
top_genes = 50 if mode == 'notebook' else int(sys.argv[8])

In [ ]:
param = str(subsample) + '_' + str(quant_min_dist) + '_' + str(quant_max_dist) + '_' + str(frac_outliers) + '_' + str(top_genes)
output_path = 'Results/' + dataset + '_' + experiment + '_' + filename.split('/')[-1].split('.')[0] + '_' + str(param) + '/'
if not os.path.exists(output_path):
    os.makedirs(output_path)

In [ ]:
nlines_list = [100] if mode == 'notebook' else range(100, 1001, 50)
seeds = range(1) if mode == 'notebook' else range(100)

baseline_methods = [KMeans, SpectralClustering, AgglomerativeClustering]
score_list = [adjusted_mutual_info_score, adjusted_rand_score]
baseline_methods_ranking = [AgglomerativeClustering]
score_list_ranking_1g = [correlation_1g, top_hits_10_1g]
score_list_ranking_2g = [correlation_2g, top_hits_10_2g]

# Synthetic Data

In [ ]:
if dataset == 'synthetic':

    ## DATASET

    X = np.loadtxt(data_path_prefix + filename)
    weights = -X[:, 2:]
    X = X[:,:2]
    X, weights = X[::subsample], weights[::subsample]
    n_pts = len(X)

    plt.figure()

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    # Plot the surface
    surf = ax.plot_trisurf(X[:,0], X[:,1], weights[:,0], cmap='viridis')

    # Labels
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

    # Optional color bar
    #fig.colorbar(surf, shrink=0.5, aspect=10)

    #plt.scatter(X[:, 0], X[:, 1], c=weights, cmap='viridis', s=10, alpha=1)
    #plt.xlabel('Coordinate 1')
    #plt.ylabel('Coordinate 2')
    #plt.grid()
    #plt.colorbar()
    #plt.title(dataset + ' - ' + str(n_pts) + ' points')
    if mode == 'notebook':
        plt.show()
    else:
        plt.savefig(output_path + dataset + '_' + experiment + '_point_cloud.png')

    ## ADJENCENCY MATRIX, SIMPLEX TREES, NEIGHBORHOOD GRAPHS, NEIGHBOR LISTS

    A = radius_neighbors_graph(X, radius=1.3, mode='connectivity', include_self=False)
    G = nx.from_scipy_sparse_array(A)
    list_neighbors = [list(G.neighbors(i)) for i in range(G.number_of_nodes())]

    ## GROUND TRUTH (TOMATO)
        
    tomato = Tomato(graph_type="manual", density_type="manual", n_clusters=6, merge_threshold=None)
    tomato.fit(list_neighbors, weights=-weights[:,0])
    ground_truth_labels = tomato.labels_

    if mode == 'notebook':
        plt.figure()
        cmap = plt.get_cmap('rainbow', ground_truth_labels.max()+1)
        bounds = np.arange(-0.5, ground_truth_labels.max() + 1.5, 1)
        norm = mcolors.BoundaryNorm(bounds, cmap.N)            
        plt.scatter(X[:, 0], X[:, 1], c=ground_truth_labels, cmap=cmap, norm=norm, s=10, alpha=1)
        plt.xlabel('Coordinate 1')
        plt.ylabel('Coordinate 2')
        plt.grid()
        plt.colorbar(ticks=np.arange(ground_truth_labels.max()+1))
        #plt.title(dataset + ' - Ground Truth Clusters (Tomato)')
        if mode == 'notebook':
            plt.show()
        else:
            plt.savefig(output_path + dataset + '_' + experiment + '_ground_truth_clusters.png')

## No Radius

In [ ]:
if dataset == 'synthetic' and experiment == 'no-radius':
    print("no radius")

    ## DATASET

    D = pairwise_distances(X)
    positive_indices = np.triu_indices_from(D, k=1)
    min_edge_length = np.quantile(D[positive_indices], quant_min_dist)
    max_edge_length = np.quantile(D[positive_indices], quant_max_dist)
    print(f"Min edge length (quantile {quant_min_dist}): {min_edge_length:.4f}")
    print(f"Max edge length (quantile {quant_max_dist}): {max_edge_length:.4f}")

    plt.figure()
    plt.hist(D[positive_indices].flatten(), bins=50)
    plt.xlabel('Distance')
    plt.ylabel('Frequency')
    plt.title('Histogram of pairwise distances')
    if mode == 'notebook':
        plt.show()
    else:
        plt.savefig(output_path + dataset + '_' + experiment+ '_pairwise_distance_histogram.png')

    ## AGNOSTIC TOMATO

    start = time.time()
    tomato_labels = []
    for distance_threshold in np.linspace(min_edge_length, max_edge_length, nlines_list[0]):
        A = radius_neighbors_graph(X, radius=distance_threshold, mode='connectivity', include_self=False)
        G = nx.from_scipy_sparse_array(A)
        list_neighbors = [list(G.neighbors(i)) for i in range(G.number_of_nodes())]
        tomato_test = Tomato(graph_type="manual", density_type="manual", n_clusters=6, merge_threshold=None)
        tomato_test.fit(list_neighbors, weights=-weights[:,0])
        labels_test = tomato_test.labels_
        tomato_labels.append(labels_test)
    end = time.time()

    tomato_time = end - start
        
    ## TOMATOMP

    tomatomp_labels = []
    tomatomp_times = []
    for nlines in nlines_list:
        start = time.time()
        tomatomp = Tomatomp(
            direction=(1.,1.),
            slice_number=nlines, 
            bounding_box=np.array([[min_edge_length, np.nan], [max_edge_length, np.nan]]), 
            merging_threshold=None, 
            n_clusters=6, 
            sigma2=0., 
            rescale=True, 
            scale_filts=[10., 1.], 
            mode='radius',
            verbose=False,
        )
        tomatomp.fit(X, weights=weights)
        conversion = {}
        for idx, l in enumerate(np.unique(tomatomp.labels_)):
            conversion[l] = idx
        labels = np.array([conversion[l] for l in tomatomp.labels_])
        end = time.time()
        tomatomp_labels.append(labels)
        tomatomp_times.append(end - start)

    plt.figure()
    cmap = plt.get_cmap('rainbow', tomatomp_labels[0].max()+1)
    bounds = np.arange(-0.5, tomatomp_labels[0].max() + 1.5, 1)
    norm = mcolors.BoundaryNorm(bounds, cmap.N)
    plt.scatter(X[:, 0], X[:, 1], c=tomatomp_labels[0], cmap=cmap, norm=norm, s=10, alpha=1)
    plt.xlabel('Coordinate 1')
    plt.ylabel('Coordinate 2')
    plt.colorbar(ticks=np.arange(tomatomp_labels[0].max()+1))
    plt.title(dataset + ' - Tomatomp Clustering')
    plt.grid()
    if mode == 'notebook':
        plt.show()
    else:
        plt.savefig(output_path + dataset + '_' + experiment + '_tomatomp_clusters.png')

    ## OTHER BASELINES (KMEANS, SPECTRAL CLUSTERING, HIERARCHICAL CLUSTERING, ETC.) CAN BE ADDED HERE

    baseline_labels = []
    baseline_times = []
    for clustering_methods in baseline_methods:
        start = time.time()
        method = clustering_methods(n_clusters=6)
        labels = method.fit_predict(X)
        end = time.time()
        baseline_time = end - start
        baseline_labels.append(labels)
        baseline_times.append(baseline_time)
    for clustering_methods in baseline_methods:
        start = time.time()
        method = clustering_methods(n_clusters=6)
        labels = method.fit_predict(np.hstack((X, weights)))
        end = time.time()
        baseline_time = end - start            
        baseline_labels.append(labels)
        baseline_times.append(baseline_time)
            
    ## SCORES

    baselines_scores = [[score(ground_truth_labels, labels) for labels in baseline_labels] for score in score_list]
    tomato_scores = [[score(ground_truth_labels, labels) for labels in tomato_labels] for score in score_list]
    tomatomp_scores = [[score(ground_truth_labels, labels) for labels in tomatomp_labels] for score in score_list]

    plt.figure()
    plt.plot(np.linspace(min_edge_length, max_edge_length, nlines_list[0]), tomato_scores[0], label='Agnostic Tomato')
    plt.axhline(tomatomp_scores[0][0], color='red', linestyle='--', label='Tomatomp')
    plt.axhline(np.min(tomato_scores[0]), color='green', linestyle='--', label='Agnostic Tomato (min)')
    plt.axhline(np.mean(tomato_scores[0]), color='blue', linestyle='--', label='Agnostic Tomato (mean)')
    plt.xlabel('Distance Threshold')
    plt.ylabel('Adjusted Mutual Information Score')
    plt.title('AMI Score vs Distance Threshold')
    plt.legend()
    if mode == 'notebook':
        plt.show()
    else:
        plt.savefig(output_path + dataset + '_' + experiment + '_ami_scores.png')

    if mode == 'notebook':
        
        for score_idx, scores in enumerate(tomatomp_scores):
            for idx, score in enumerate(scores):
                print(f"Tomatomp (nlines = {nlines_list[idx]}) {score_list[score_idx].__name__} Score: {score:.4f}")
        for score_idx, score in enumerate(tomato_scores):
            print(f"Agnostic Tomato (min) {score_list[score_idx].__name__} Score: {np.min(score):.4f}")
            print(f"Agnostic Tomato (mean) {score_list[score_idx].__name__} Score: {np.mean(score):.4f}")
        for score_idx, scores in enumerate(baselines_scores):
            for idx, score in enumerate(scores):
                print(f"Baseline {idx+1} {score_list[score_idx].__name__} Score: {score:.4f}")
            
        for idx, tomatomp_time in enumerate(tomatomp_times):
            print(f"Tomatomp (nlines = {nlines_list[idx]}) Time: {tomatomp_time:.2f} seconds")
        print(f"Agnostic Tomato Time: {tomato_time:.2f} seconds")
        for idx, baseline_time in enumerate(baseline_times):
            print(f"Baseline {idx+1} Time: {baseline_time:.2f} seconds")
        
    else:

        with open(output_path + dataset + '_' + experiment + '_ami_scores.txt', 'w') as f:
            for score_idx, scores in enumerate(tomatomp_scores):
                for idx, score in enumerate(scores):
                    f.write(f"Tomatomp (nlines = {nlines_list[idx]}) {score_list[score_idx].__name__} Score: {score:.4f}\n")
            for score_idx, score in enumerate(tomato_scores):
                f.write(f"Agnostic Tomato (min) {score_list[score_idx].__name__} Score: {np.min(score):.4f}\n")
                f.write(f"Agnostic Tomato (mean) {score_list[score_idx].__name__} Score: {np.mean(score):.4f}\n")
            for score_idx, scores in enumerate(baselines_scores):
                for idx, score in enumerate(scores):
                    f.write(f"Baseline {idx+1} {score_list[score_idx].__name__} Score: {score:.4f}\n")
            
        with open(output_path + dataset + '_' + experiment + '_times.txt', 'w') as f:
            for idx, tomatomp_time in enumerate(tomatomp_times):
                f.write(f"Tomatomp (nlines = {nlines_list[idx]}) Time: {tomatomp_time:.2f} seconds\n")
            f.write(f"Agnostic Tomato Time: {tomato_time:.2f} seconds\n")
            for idx, baseline_time in enumerate(baseline_times):
                f.write(f"Baseline {idx+1} Time: {baseline_time:.2f} seconds\n")

## Outliers

In [ ]:
if dataset == 'synthetic' and experiment == 'outliers':
    print("outliers")
    
    for seed in seeds:

        ## DATASET

        np.random.seed(seed)
        n_outliers = int(frac_outliers * n_pts)
        outliers = np.random.uniform(low=X.min(), high=X.max(), size=(n_outliers, 2))
        outlier_weights = np.random.uniform(low=weights.min()-1000, high=weights.min()-900, size=(n_outliers, 1))
        X_mod = np.vstack((X, outliers))
        weights_mod = np.vstack((weights, outlier_weights))
        ground_truth_labels_mod = np.hstack((ground_truth_labels, [-1]*n_outliers))

        plt.figure()
        plt.scatter(X_mod[:, 0], X_mod[:, 1], c=weights_mod, cmap='viridis', s=10, alpha=1)
        plt.scatter(outliers[:, 0], outliers[:, 1], c='red', s=50, alpha=1, label='Outliers')
        plt.xlabel('Coordinate 1')
        plt.ylabel('Coordinate 2')
        plt.grid()
        plt.colorbar()
        #plt.title(dataset + ' with Outliers - ' + str(n_pts) + ' points + ' + str(n_outliers) + ' outliers')
        if mode == 'notebook':
            plt.show()
        else:
            plt.savefig(output_path + dataset + '_' + experiment + '_point_cloud_seed_' + str(seed) + '.png')
        
        ## ADJENCENCY MATRIX, SIMPLEX TREES, NEIGHBORHOOD GRAPHS, NEIGHBOR LISTS

        A = radius_neighbors_graph(X_mod, radius=1.3, mode='connectivity', include_self=False)
        G = nx.from_scipy_sparse_array(A)
        list_neighbors = [list(G.neighbors(i)) for i in range(G.number_of_nodes())]

        #A = NearestNeighbors(n_neighbors=8).fit(X_mod).kneighbors_graph(X_mod)
        #G = nx.from_numpy_array(A)
        #list_neighbors = [list(G.neighbors(i)) for i in range(G.number_of_nodes())]

        ## TOMATO

        start = time.time()
        tomato = Tomato(graph_type='manual', density_type='manual', n_clusters=6, merge_threshold=None)
        tomato.fit(list_neighbors, weights=-weights_mod.flatten())
        end = time.time()

        tomato_labels = tomato.labels_
        tomato_time = end - start

        plt.figure()
        cmap = plt.get_cmap('rainbow', tomato_labels.max()+1)
        bounds = np.arange(-0.5, tomato_labels.max() + 1.5, 1)
        norm = mcolors.BoundaryNorm(bounds, cmap.N)
        plt.scatter(X_mod[:, 0], X_mod[:, 1], c=tomato_labels, cmap=cmap, norm=norm, s=10, alpha=1)
        plt.xlabel('Coordinate 1')
        plt.ylabel('Coordinate 2')
        plt.grid()
        plt.colorbar(ticks=np.arange(tomato_labels.max()+1))
        #plt.title(dataset + ' with Outliers - Tomato Clustering')
        if mode == 'notebook':
            plt.show()
        else:
            plt.savefig(output_path + dataset + '_' + experiment + '_tomato_clusters_seed_' + str(seed) + '.png')

        ## TOMATOMP

        outlier_scores = np.zeros(len(X_mod))
        for i, list_neighbor in enumerate(list_neighbors):
            if len(list_neighbor) == 0:
                outlier_scores[i] = 0
            else:
                outlier_scores[i] = np.mean([np.abs(weights_mod[i]-weights_mod[n]) for n in list_neighbor])

        plt.figure()
        plt.scatter(X_mod[:, 0], X_mod[:, 1], c=outlier_scores, cmap='viridis', s=10, alpha=1)
        plt.xlabel('Coordinate 1')
        plt.ylabel('Coordinate 2')
        plt.grid()
        plt.colorbar()
        #plt.title(dataset + ' - Outlier Scores')
        if mode == 'notebook':
            plt.show()
        else:
            plt.savefig(output_path + dataset + '_' + experiment + '_outlier_scores_seed_' + str(seed) + '.png')

        tomatomp_labels = []
        tomatomp_times = []
        for nlines in nlines_list:
            start = time.time()
            tomatomp = Tomatomp(
                direction=(1.,1.),
                slice_number=nlines, 
                bounding_box=np.array([[np.nan, np.nan], [np.nan, np.nan]]),
                #bounding_box=np.array([[-outlier_scores.max(), np.nan], [np.nan, np.nan]]), 
                merging_threshold=None, 
                n_clusters=6, 
                sigma2=0., 
                rescale=False,
                #scale_filts=[10., 1.], 
                #mode='radius',
                verbose=False,
            )
            tomatomp.fit(G, weights=np.hstack([weights_mod, outlier_scores[:,None]]))
            conversion = {}
            for idx, l in enumerate(np.unique(tomatomp.labels_)):
                conversion[l] = idx
            labels = np.array([conversion[l] for l in tomatomp.labels_])
            end = time.time()
            tomatomp_labels.append(labels)
            tomatomp_times.append(end - start)

        plt.figure()
        cmap = plt.get_cmap('rainbow', tomatomp_labels[0].max()+1)
        bounds = np.arange(-0.5, tomatomp_labels[0].max() + 1.5, 1)
        norm = mcolors.BoundaryNorm(bounds, cmap.N)
        plt.scatter(X_mod[:, 0], X_mod[:, 1], c=tomatomp_labels[0], cmap=cmap, norm=norm, s=10, alpha=1)
        plt.xlabel('Coordinate 1')
        plt.ylabel('Coordinate 2')
        plt.colorbar(ticks=np.arange(tomatomp_labels[0].max()+1))
        #plt.title(dataset + ' - Tomatomp Clustering')
        plt.grid()
        if mode == 'notebook':
            plt.show()
        else:
            plt.savefig(output_path + dataset + '_' + experiment + '_tomatomp_clusters.png')

        ## OTHER BASELINES (KMEANS, SPECTRAL CLUSTERING, HIERARCHICAL CLUSTERING, ETC.)

        baseline_labels = []
        baseline_times = []
        for clustering_methods in baseline_methods:
            start = time.time()
            method = clustering_methods(n_clusters=6)
            labels = method.fit_predict(X_mod)
            end = time.time()
            baseline_time = end - start
            baseline_labels.append(labels)
            baseline_times.append(baseline_time)
        for clustering_methods in baseline_methods:
            start = time.time()
            method = clustering_methods(n_clusters=6)
            labels = method.fit_predict(np.hstack((X_mod, weights_mod, outlier_scores[:,None])))
            end = time.time()
            baseline_time = end - start            
            baseline_labels.append(labels)
            baseline_times.append(baseline_time)

        ## SCORES

        baselines_scores = [[score(ground_truth_labels_mod, labels) for labels in baseline_labels] for score in score_list]
        tomato_scores = [score(ground_truth_labels_mod, tomato_labels) for score in score_list]
        tomatomp_scores = [[score(ground_truth_labels_mod, labels) for labels in tomatomp_labels] for score in score_list]

        if mode == 'notebook':
            
            for score_idx, scores in enumerate(tomatomp_scores):
                for idx, score in enumerate(scores):
                    print(f"Tomatomp (nlines = {nlines_list[idx]}) {score_list[score_idx].__name__} Score (seed {seed}): {score:.4f}")
            for score_idx, score in enumerate(tomato_scores):
                print(f"Tomato {score_list[score_idx].__name__} Score (seed {seed}): {score:.4f}")
            for score_idx, scores in enumerate(baselines_scores):
                for idx, score in enumerate(scores):
                    print(f"Baseline {idx+1} {score_list[score_idx].__name__} Score (seed {seed}): {score:.4f}")
            
            for idx, tomatomp_time in enumerate(tomatomp_times):
                print(f"Tomatomp (nlines = {nlines_list[idx]}) Time (seed {seed}): {tomatomp_time:.2f} seconds")
            print(f"Tomato Time (seed {seed}): {tomato_time:.2f} seconds")
            for idx, baseline_time in enumerate(baseline_times):
                print(f"Baseline {idx+1} Time (seed {seed}): {baseline_time:.2f} seconds")
        
        else:

            with open(output_path + dataset + '_' + experiment + f'_ami_scores_seed{seed}.txt', 'w') as f:
                for score_idx, scores in enumerate(tomatomp_scores):
                    for idx, score in enumerate(scores):
                        f.write(f"Tomatomp (nlines = {nlines_list[idx]}) {score_list[score_idx].__name__} Score (seed {seed}): {score:.4f}\n")
                for score_idx, score in enumerate(tomato_scores):
                    f.write(f"Tomato {score_list[score_idx].__name__} Score (seed {seed}): {score:.4f}\n")
                for score_idx, scores in enumerate(baselines_scores):
                    for idx, score in enumerate(scores):
                        f.write(f"Baseline {idx+1} {score_list[score_idx].__name__} Score (seed {seed}): {score:.4f}\n")
            
            with open(output_path + dataset + '_' + experiment + f'_times_seed{seed}.txt', 'w') as f:
                for idx, tomatomp_time in enumerate(tomatomp_times):
                    f.write(f"Tomatomp (nlines = {nlines_list[idx]}) Time (seed {seed}): {tomatomp_time:.2f} seconds\n")
                f.write(f"Tomato Time (seed {seed}): {tomato_time:.2f} seconds\n")
                for idx, baseline_time in enumerate(baseline_times):
                    f.write(f"Baseline {idx+1} Time (seed {seed}): {baseline_time:.2f} seconds\n")

# Image Data

In [ ]:
if dataset == 'image' and experiment == 'outliers':
    print("outliers")

    ## DATASET
    
    image = data_path_prefix + filename
    X = plt.imread(image)
    X = np.array(X, dtype=np.float64)
    print(f"Image shape: {X.shape}")
    
    plt.figure()
    plt.imshow(X, cmap='gray')
    #plt.title('Original image')
    plt.axis('off')
    plt.colorbar()
    if mode == 'notebook':
        plt.show()
    else:
        plt.savefig(output_path + dataset + '_' + experiment + '_original_image.png')

    ## ADJENCENCY MATRIX, SIMPLEX TREES, NEIGHBORHOOD GRAPHS, NEIGHBOR LISTS

    list_neighbors = []
    greylevel = []
    pixel = 0
    st = gd.SimplexTree()
    for idp in range(X.shape[0]):
        for idq in range(X.shape[1]):
            for potential_neighbor in [pixel-1, pixel+1, pixel-X.shape[1], pixel+X.shape[1], pixel-X.shape[1]-1, pixel+X.shape[1]-1, pixel-X.shape[1]+1, pixel+X.shape[1]+1]:
                if 0 <= potential_neighbor < X.size:
                    st.insert([pixel, potential_neighbor])
            greylevel.append(float(X[idp, idq]))
            pixel += 1

    n_vertices = X.shape[0] * X.shape[1]
    A = np.zeros((n_vertices, n_vertices))
    for splx in st.get_skeleton(1):
        if len(splx[0]) == 2:
            A[splx[0][0], splx[0][1]] = 1
            A[splx[0][1], splx[0][0]] = 1
    G = nx.from_numpy_array(A)
    list_neighbors = [list(G.neighbors(i)) for i in range(A.shape[0])]

    tomato = Tomato(graph_type='manual', density_type='manual', n_clusters=12, merge_threshold=None)
    tomato.fit(X=list_neighbors, weights=np.array(greylevel))
    ground_truth_labels = tomato.labels_

    plt.figure()
    cmap = plt.get_cmap('rainbow', ground_truth_labels.max()+1)
    bounds = np.arange(-0.5, ground_truth_labels.max() + 1.5, 1)
    norm = mcolors.BoundaryNorm(bounds, cmap.N)
    plt.imshow(X, cmap='gray', alpha=0.9)
    plt.imshow(ground_truth_labels.reshape(X.shape), cmap=cmap, norm=norm, alpha=0.75)
    #plt.title('Ground Truth Clusters (Tomato)')
    plt.axis('off')
    plt.colorbar(ticks=np.arange(ground_truth_labels.max()+1))
    if mode == 'notebook':
        plt.show()
    else:
        plt.savefig(output_path + dataset + '_' + experiment + '_ground_truth_clusters.png')

    n_outliers = int(frac_outliers * X.shape[0] * X.shape[1])

    for seed in seeds:
    
        np.random.seed(seed)
        outliers_x = np.random.choice(X.shape[0], size=n_outliers, replace=False)
        outliers_y = np.random.choice(X.shape[1], size=n_outliers, replace=False)
        Xmax = X.max()

        Xmod = X.copy()
        Xmod += np.random.normal(scale=0.001, size=X.shape)
        for ido,_ in enumerate(outliers_x):
            Xmod[outliers_x[ido], outliers_y[ido]] = Xmax

        greylevel = []
        for idp in range(Xmod.shape[0]):
            for idq in range(Xmod.shape[1]):
                greylevel.append(float(Xmod[idp, idq]))

        plt.figure()
        plt.imshow(Xmod, cmap='gray')
        #plt.title('Image with outliers')
        plt.axis('off')
        plt.colorbar()
        if mode == 'notebook':
            plt.show()
        else:
            plt.savefig(output_path + dataset + '_' + experiment + '_image_with_outliers_seed_' + str(seed) + '.png')

        ## TOMATO

        start = time.time()
        tomato = Tomato(graph_type='manual', density_type='manual', n_clusters=12, merge_threshold=None)
        tomato.fit(list_neighbors, weights=np.array(greylevel))
        end = time.time()

        tomato_labels = tomato.labels_
        tomato_time = end - start

        plt.figure()
        cmap = plt.get_cmap('rainbow', tomato_labels.max()+1)
        bounds = np.arange(-0.5, tomato_labels.max() + 1.5, 1)
        norm = mcolors.BoundaryNorm(bounds, cmap.N)
        plt.imshow(tomato_labels.reshape(Xmod.shape), cmap=cmap, norm=norm, alpha=0.5)
        plt.colorbar(ticks=np.arange(tomato_labels.max()+1))
        plt.imshow(Xmod, cmap='gray', alpha=0.5)
        #plt.title('Tomato clustering')
        plt.axis('off')
        if mode == 'notebook':
            plt.show()
        else:
            plt.savefig(output_path + dataset + '_' + experiment + '_tomato_clusters_seed_' + str(seed) + '.png')

        ## TOMATOMP

        outlier_scores = np.array([np.array([np.abs(greylevel[n]-greylevel[i]) for n in lneighb]).mean() for i, lneighb in enumerate(list_neighbors)])
        weights = np.hstack([outlier_scores[:,None], -np.array(greylevel)[:,None]])

        plt.figure()
        plt.imshow(Xmod, alpha=0.25)
        plt.xlabel('Coordinate 1')
        plt.ylabel('Coordinate 2')
        plt.grid()
        plt.colorbar()
        #plt.title(dataset + ' - Outlier Scores')
        if mode == 'notebook':
            plt.show()
        else:
            plt.savefig(output_path + dataset + '_' + experiment + '_outlier_scores_seed_' + str(seed) + '.png')

        tomatomp_labels = []
        tomatomp_times = []
        for nlines in nlines_list:
            start = time.time()
            model = Tomatomp(
                    direction=(1., 1.),
                    slice_number=nlines,
                    bounding_box=np.array([[np.nan, np.nan], [np.nan, np.nan]]),
                    merging_threshold=None,
                    n_clusters=12,
                    sigma2=0.,
                    rescale=True,
                    scale_filts=[10., 1.],
                    verbose=False,
            )
            model.fit(G, weights=weights)
            end = time.time()
            labels = np.array(model.labels_)
            tomatomp_labels.append(labels)
            tomatomp_times.append(end - start)

        n_classes = tomatomp_labels[0].max() + 1
        clus_sizes = [np.sum(tomatomp_labels[0] == i) for i in range(n_classes)]
        ranks = np.zeros(n_classes)
        clus_order = np.argsort(clus_sizes)[::-1]
        for i in range(len(clus_order)):
            clus = clus_order[i]
            ranks[clus] = i
        maxcluster = 30
        labels_plot = np.array([ranks[lab] if lab in clus_order[:maxcluster] else maxcluster+1 for lab in tomatomp_labels[0]])

        plt.figure()
        cmap = plt.get_cmap('rainbow', labels_plot.max()+1)
        bounds = np.arange(-0.5, labels_plot.max() + 1.5, 1)
        norm = mcolors.BoundaryNorm(bounds, cmap.N)
        plt.imshow(labels_plot.reshape(Xmod.shape), cmap=cmap, norm=norm)
        plt.colorbar(ticks=np.arange(labels_plot.max()+1))
        plt.imshow(Xmod, cmap="gray", alpha=0.25)
        #plt.title('Tomatomp clustering')
        plt.axis('off')
        if mode == 'notebook':
            plt.show()
        else:
            plt.savefig(output_path + dataset + '_' + experiment + '_tomatomp_clusters_seed_' + str(seed) + '.png')

        ## BASELINES (KMEANS, SPECTRAL CLUSTERING, HIERARCHICAL CLUSTERING, ETC.)
        
        baseline_labels = []
        baseline_times = []
        point_cloud = np.array([[i,j,Xmod[i,j]] for i in range(Xmod.shape[0]) for j in range(Xmod.shape[1])])
        baseline_methods = [KMeans, AgglomerativeClustering]
        for clustering_methods in baseline_methods:
            start = time.time()
            method = clustering_methods(n_clusters=12)
            labels = method.fit_predict(point_cloud)
            end = time.time()
            baseline_time = end - start
            baseline_labels.append(labels)
            baseline_times.append(baseline_time)
        for clustering_methods in baseline_methods:
            start = time.time()
            method = clustering_methods(n_clusters=12)
            labels = method.fit_predict(np.hstack((point_cloud, outlier_scores[:,None])))
            end = time.time()
            baseline_time = end - start            
            baseline_labels.append(labels)
            baseline_times.append(baseline_time)

        ## SCORES

        baselines_scores = [[score(ground_truth_labels, labels) for labels in baseline_labels] for score in score_list]
        tomato_scores = [score(ground_truth_labels, tomato_labels) for score in score_list]
        tomatomp_scores = [[score(ground_truth_labels, labels) for labels in tomatomp_labels] for score in score_list]

        if mode == 'notebook':

            for score_idx, scores in enumerate(tomatomp_scores):
                for idx, score in enumerate(scores):
                    print(f"Tomatomp (nlines = {nlines_list[idx]}) {score_list[score_idx].__name__} Score (seed {seed}): {score:.4f}")
            for score_idx, score in enumerate(tomato_scores):
                print(f"Tomato {score_list[score_idx].__name__} Score (seed {seed}): {score:.4f}")
            for score_idx, scores in enumerate(baselines_scores):
                for idx, score in enumerate(scores):
                    print(f"Baseline {idx+1} {score_list[score_idx].__name__} Score (seed {seed}): {score:.4f}")
            
            for idx, tomatomp_time in enumerate(tomatomp_times):
                print(f"Tomatomp (nlines = {nlines_list[idx]}) Time (seed {seed}): {tomatomp_time:.2f} seconds")
            print(f"Tomato Time (seed {seed}): {tomato_time:.2f} seconds")
            for idx, baseline_time in enumerate(baseline_times):
                print(f"Baseline {idx+1} Time (seed {seed}): {baseline_time:.2f} seconds")
                
        else:

            with open(output_path + dataset + '_' + experiment + f'_ami_scores_seed{seed}.txt', 'w') as f:
                for score_idx, scores in enumerate(tomatomp_scores):
                    for idx, score in enumerate(scores):
                        f.write(f"Tomatomp (nlines = {nlines_list[idx]}) {score_list[score_idx].__name__} Score (seed {seed}): {score:.4f}\n")
                for score_idx, score in enumerate(tomato_scores):
                    f.write(f"Tomato {score_list[score_idx].__name__} Score (seed {seed}): {score:.4f}\n")
                for score_idx, scores in enumerate(baselines_scores):
                    for idx, score in enumerate(scores):
                        f.write(f"Baseline {idx+1} {score_list[score_idx].__name__} Score (seed {seed}): {score:.4f}\n")
            
            with open(output_path + dataset + '_' + experiment + f'_times_seed{seed}.txt', 'w') as f:
                for idx, tomatomp_time in enumerate(tomatomp_times):
                    f.write(f"Tomatomp (nlines = {nlines_list[idx]}) Time (seed {seed}): {tomatomp_time:.2f} seconds\n")
                f.write(f"Tomato Time (seed {seed}): {tomato_time:.2f} seconds\n")
                for idx, baseline_time in enumerate(baseline_times):
                    f.write(f"Baseline {idx+1} Time (seed {seed}): {baseline_time:.2f} seconds\n")

# 3D Shape Data

In [ ]:
if dataset == '3dshape' and experiment == 'no-radius':
    print("no radius")

    ## DATASET
         
    X, F = read_off(path=data_path_prefix + filename)
    n_pts = X.shape[0]
    edges = set()
    for a, b, c in F:
        edges.add(tuple(sorted((a, b))))
        edges.add(tuple(sorted((b, c))))
        edges.add(tuple(sorted((c, a))))

    rows, cols, vals = [], [], []
    for i, j in edges:
        rows.extend([i, j])
        cols.extend([j, i])
        vals.extend([1, 1])
        
    print(f"Number of points: {n_pts}")
    print(f"Number of edges: {len(edges)}")
    
    ## ADJENCENCY MATRIX, SIMPLEX TREES, NEIGHBORHOOD GRAPHS, NEIGHBOR LISTS

    A = csr_matrix((vals, (rows, cols)), shape=(n_pts, n_pts))
    G = nx.from_scipy_sparse_array(A)
    list_neighbors = [list(G.neighbors(i)) for i in range(n_pts)]
    
    deg = np.array(A.sum(axis=1)).ravel()
    D = diags(deg)
    L = D - A
    eigvals, eigvecs = eigsh(L, k=200, M=D, sigma=0.0, which="LM")
    order = np.argsort(eigvals)
    eigvecs = eigvecs[:, order]
    hksvals = hks(eigvals, eigvecs, 1000)

    ## GROUND TRUTH (TOMATO)

    tomato = Tomato(graph_type='manual', density_type='manual', n_clusters=5, merge_threshold=None)
    tomato.fit(X=list_neighbors, weights=hksvals)
    ground_truth_labels = tomato.labels_

    if mode == 'notebook':
        plt.figure()
        mp.plot(X, F, c=ground_truth_labels, shading={"point_size": 5})
        plt.show()

    X = X[::subsample]
    ground_truth_labels = ground_truth_labels[::subsample]
    hksvals = hksvals[::subsample]
    
    D = pairwise_distances(X)
    positive_indices = np.triu_indices_from(D, k=1)
    min_edge_length = np.quantile(D[positive_indices], quant_min_dist)
    max_edge_length = np.quantile(D[positive_indices], quant_max_dist)
    print(f"Min edge length (quantile {quant_min_dist}): {min_edge_length:.4f}")
    print(f"Max edge length (quantile {quant_max_dist}): {max_edge_length:.4f}")

    n_pts = X.shape[0]
    
    ## AGNOSTIC TOMATO

    start = time.time()
    tomato_labels = []
    for distance_threshold in np.linspace(min_edge_length, max_edge_length, nlines_list[0]):
        A = radius_neighbors_graph(X, radius=distance_threshold, mode='connectivity', include_self=False)
        G = nx.from_scipy_sparse_array(A)
        list_neighbors = [list(G.neighbors(i)) for i in range(G.number_of_nodes())]
        tomato_test = Tomato(graph_type='manual', density_type='manual', n_clusters=5, merge_threshold=None)
        tomato_test.fit(list_neighbors, weights=hksvals)
        labels_test = tomato_test.labels_
        tomato_labels.append(labels_test)
    end = time.time()

    tomato_time = end - start

    ## TOMATOMP

    tomatomp_labels = []
    tomatomp_times = []
    for nlines in nlines_list:
        start = time.time()
        tomatomp = Tomatomp(
            direction=(1., 1.),
            slice_number=nlines,
            bounding_box=np.array([[min_edge_length, np.nan], [max_edge_length, np.nan]]),
            merging_threshold=None,
            n_clusters=5,
            sigma2=0.,
            rescale=True,
            scale_filts=[10., 1.],
            mode='radius',
            verbose=False,
        )
        tomatomp.fit(X, weights=-hksvals[:,None])
        conversion = {}
        for idx, l in enumerate(np.unique(tomatomp.labels_)):
            conversion[l] = idx
        labels = np.array([conversion[l] for l in tomatomp.labels_])
        end = time.time()
        tomatomp_labels.append(labels)
        tomatomp_times.append(end - start)

    if mode == 'notebook':
        plt.figure()
        mp.plot(X, F, c=tomatomp_labels[0], shading={"point_size": 5})
        plt.show()

    ## OTHER BASELINES (KMEANS, SPECTRAL CLUSTERING, HIERARCHICAL CLUSTERING, ETC.)

    baseline_labels = []
    baseline_times = []
    for clustering_methods in baseline_methods:
        start = time.time()
        method = clustering_methods(n_clusters=5)
        labels = method.fit_predict(X)
        end = time.time()
        baseline_time = end - start
        baseline_labels.append(labels)
        baseline_times.append(baseline_time)
    for clustering_methods in baseline_methods:
        start = time.time()
        method = clustering_methods(n_clusters=5)
        labels = method.fit_predict(np.hstack((X, hksvals[:,None])))
        end = time.time()
        baseline_time = end - start            
        baseline_labels.append(labels)
        baseline_times.append(baseline_time)

    ## SCORES

    baselines_scores = [[score(ground_truth_labels, labels) for labels in baseline_labels] for score in score_list]
    tomato_scores = [[score(ground_truth_labels, labels) for labels in tomato_labels] for score in score_list]
    tomatomp_scores = [[score(ground_truth_labels, labels) for labels in tomatomp_labels] for score in score_list]

    plt.figure()
    plt.plot(np.linspace(min_edge_length, max_edge_length, nlines_list[0]), tomato_scores[0], label='Agnostic Tomato')
    plt.axhline(tomatomp_scores[0][0], color='red', linestyle='--', label='Tomatomp')
    plt.axhline(np.min(tomato_scores[0]), color='green', linestyle='--', label='Agnostic Tomato (min)')
    plt.axhline(np.mean(tomato_scores[0]), color='blue', linestyle='--', label='Agnostic Tomato (mean)')
    plt.xlabel('Distance Threshold')
    plt.ylabel('Adjusted Mutual Information Score')
    plt.title('AMI Score vs Distance Threshold')
    plt.legend()
    if mode == 'notebook':
        plt.show()
    else:
        plt.savefig(output_path + dataset + '_' + experiment + '_ami_scores.png')

    if mode == 'notebook':

        for score_idx, scores in enumerate(tomatomp_scores):
            for idx, score in enumerate(scores):
                print(f"Tomatomp (nlines = {nlines_list[idx]}) {score_list[score_idx].__name__} Score: {score:.4f}")
        for score_idx, score in enumerate(tomato_scores):
            print(f"Agnostic Tomato (min) {score_list[score_idx].__name__} Score: {np.min(score):.4f}")
            print(f"Agnostic Tomato (mean) {score_list[score_idx].__name__} Score: {np.mean(score):.4f}")
        for score_idx, scores in enumerate(baselines_scores):
            for idx, score in enumerate(scores):
                print(f"Baseline {idx+1} {score_list[score_idx].__name__} Score: {score:.4f}")
            
        for idx, tomatomp_time in enumerate(tomatomp_times):
            print(f"Tomatomp (nlines = {nlines_list[idx]}) Time: {tomatomp_time:.2f} seconds")
        print(f"Agnostic Tomato Time: {tomato_time:.2f} seconds")
        for idx, baseline_time in enumerate(baseline_times):
            print(f"Baseline {idx+1} Time: {baseline_time:.2f} seconds")

    else:

        with open(output_path + dataset + '_' + experiment + '_ami_scores.txt', 'w') as f:
            for score_idx, scores in enumerate(tomatomp_scores):
                for idx, score in enumerate(scores):
                    f.write(f"Tomatomp (nlines = {nlines_list[idx]}) {score_list[score_idx].__name__} Score: {score:.4f}\n")
            for score_idx, score in enumerate(tomato_scores):
                f.write(f"Agnostic Tomato (min) {score_list[score_idx].__name__} Score: {np.min(score):.4f}\n")
                f.write(f"Agnostic Tomato (mean) {score_list[score_idx].__name__} Score: {np.mean(score):.4f}\n")
            for score_idx, scores in enumerate(baselines_scores):
                for idx, score in enumerate(scores):
                    f.write(f"Baseline {idx+1} {score_list[score_idx].__name__} Score: {score:.4f}\n")
            
        with open(output_path + dataset + '_' + experiment + '_times.txt', 'w') as f:
            for idx, tomatomp_time in enumerate(tomatomp_times):
                f.write(f"Tomatomp (nlines = {nlines_list[idx]}) Time: {tomatomp_time:.2f} seconds\n")
            f.write(f"Agnostic Tomato Time: {tomato_time:.2f} seconds\n")
            for idx, baseline_time in enumerate(baseline_times):
                f.write(f"Baseline {idx+1} Time: {baseline_time:.2f} seconds\n")

# ST 1 Gene Ranking

In [ ]:
if dataset == 'st1gr' and filename.endswith('.csv'):

    rescale = True

    ## DATASET

    df = pd.read_csv(data_path_prefix + filename)
    spatial_coords = np.array(df[['x_position', 'y_position']].values)
    gene_expression = np.array(df.drop(columns=['x_position', 'y_position']).values)
    dtms, st, A, G, list_neighbors = smoothed_expression(df, mesh_type="hexagonal", m=0.1)

In [ ]:
if dataset == 'st1gr' and filename.endswith('.npy'):
    
    rescale = False
    
    ## DATASET
    
    spatial_coords = np.load(data_path_prefix + filename)
    dtms = pck.load(open(data_path_prefix + filename.replace('spatial_coords.npy', 'spatial_dtms.pkl'), 'rb'))
    jaccard_ranking = pck.load(open(data_path_prefix + filename.replace('spatial_coords.npy', 'jaccard_results.pkl'), 'rb'))

    spatial_coords = spatial_coords[::subsample]
    dtms = [(idx_g, gene, values[::subsample]) for idx_g, gene, values in dtms]
    A = radius_neighbors_graph(spatial_coords, radius=2.5, mode='connectivity', include_self=False)
    G = nx.from_scipy_sparse_array(A)
    list_neighbors = [list(G.neighbors(i)) for i in range(G.number_of_nodes())]

In [ ]:
if dataset == 'st1gr':

    ## DATASET
    
    print(len(spatial_coords), len(dtms))
    print(dtms[1][1])

    dtm_global_min = min([dtm.min() for (_, _, dtm) in dtms])
    dtm_global_max = max([dtm.max() for (_, _, dtm) in dtms])

    plt.figure()
    plt.scatter(spatial_coords[:, 0], spatial_coords[:, 1], c=dtms[1][2], cmap='viridis', s=10, alpha=1)
    plt.xlabel('X Coordinate')
    plt.ylabel('Y Coordinate')
    plt.grid()
    plt.axis('equal')
    plt.colorbar()
    #plt.title('Spatial Coordinates Colored by First Gene Expression')
    if mode == 'notebook':
        plt.show()
    else:
        plt.savefig(output_path + dataset + '_spatial_coordinates.png')

    ## GROUND TRUTH (TOMATO)

    ground_truth_ranking = rank_genes_tomato(dtms, list_neighbors, merge_threshold=None)
    
    top_idxs = np.argsort(ground_truth_ranking['score'])[::-1][:top_genes]
    ground_truth_ranking = ground_truth_ranking.iloc[top_idxs]
    dtms = [dtms[idx] for idx in top_idxs]

    if mode == 'notebook':
        print(ground_truth_ranking.head(50))
    else:
        with open(output_path + dataset + '_ground_truth_ranking.txt', 'w') as f:
            for idx, gene in enumerate(ground_truth_ranking['gene']):
                f.write(f"{idx+1}. {gene}\n")

In [ ]:
#plt.figure()
#plt.hist(pairwise_distances(spatial_coords)[np.triu_indices(len(spatial_coords), k=1)], bins=100)
#plt.show()

## No Radius

In [ ]:
if dataset == 'st1gr' and experiment == 'no-radius':
    print("no-radius")

    min_edge_length = quant_min_dist
    max_edge_length = quant_max_dist

    ## AGNOSTIC TOMATO

    start = time.time()
    tomato_rankings = []
    for distance_threshold in np.linspace(min_edge_length, max_edge_length, nlines_list[0]):
        A = radius_neighbors_graph(spatial_coords, radius=distance_threshold, mode='connectivity', include_self=False)
        G = nx.from_scipy_sparse_array(A)
        list_neighbors = [list(G.neighbors(i)) for i in range(G.number_of_nodes())]
        tomato_ranking_test = rank_genes_tomato(dtms, list_neighbors, merge_threshold=None)
        tomato_rankings.append(tomato_ranking_test)
    end = time.time()

    tomato_time = end - start

    ## TOMATOMP
    
    tomatomp_rankings = []
    tomatomp_times = []
    for nlines in nlines_list:
        start = time.time()
        ranking1, ranking2, ranking3, ranking4, ranking5, ranking6 = rank_genes_tomatomp_radius(dtms, spatial_coords, direction=(1., 1.), nlines=nlines, min_edge_length=min_edge_length, max_edge_length=max_edge_length, rescale=rescale)
        tomatomp_rankings.append(ranking1)
        tomatomp_rankings.append(ranking2)
        tomatomp_rankings.append(ranking3)
        tomatomp_rankings.append(ranking4)
        tomatomp_rankings.append(ranking5)
        tomatomp_rankings.append(ranking6)
        end = time.time()
        tomatomp_times.append(end - start)

    ## BASELINES (KMEANS, SPECTRAL CLUSTERING, HIERARCHICAL CLUSTERING, ETC.)

    baseline_rankings = []
    baseline_times = []
    for ranking_methods in baseline_methods_ranking:
        start = time.time()
        ranking = rank_genes_hierarchical(dtms, spatial_coords, add_outlier_score=False)
        baseline_rankings.append(ranking)
        end = time.time()
        baseline_times.append(end - start)

    ## SCORES (CORRELATION, MUTUAL INFORMATION, ETC.)

    baselines_scores = [[score(ground_truth_ranking, ranking) for ranking in baseline_rankings] for score in score_list_ranking_1g]
    tomato_scores = [[score(ground_truth_ranking, ranking) for ranking in tomato_rankings] for score in score_list_ranking_1g]
    tomatomp_scores = [[score(ground_truth_ranking, ranking) for ranking in tomatomp_rankings] for score in score_list_ranking_1g]

    if mode == 'notebook':
        
        for score_idx, scores in enumerate(tomatomp_scores):
            for idx, score in enumerate(scores):
                print(f"Tomatomp (nlines = {nlines_list[idx//6]}, mode {idx%6+1}) {score_list_ranking_1g[score_idx].__name__} Score: {score:.4f}")
        for score_idx, score in enumerate(tomato_scores):
            print(f"Agnostic Tomato (min) {score_list_ranking_1g[score_idx].__name__} Score: {np.min(score):.4f}")
            print(f"Agnostic Tomato (mean) {score_list_ranking_1g[score_idx].__name__} Score: {np.mean(score):.4f}")
        for score_idx, scores in enumerate(baselines_scores):
            for idx, score in enumerate(scores):
                print(f"Baseline {idx+1} {score_list_ranking_1g[score_idx].__name__} Score: {score:.4f}")
            
        for idx, tomatomp_time in enumerate(tomatomp_times):
            print(f"Tomatomp (nlines = {nlines_list[idx]}) Time: {tomatomp_time:.2f} seconds")
        print(f"Agnostic Tomato Time: {tomato_time:.2f} seconds")
        for idx, baseline_time in enumerate(baseline_times):
            print(f"Baseline {idx+1} Time: {baseline_time:.2f} seconds")

    else:

        with open(output_path + dataset + '_' + experiment + '_ranking_scores.txt', 'w') as f:
            for score_idx, scores in enumerate(tomatomp_scores):
                for idx, score in enumerate(scores):
                    f.write(f"Tomatomp (nlines = {nlines_list[idx//6]}, mode {idx%6+1}) {score_list_ranking_1g[score_idx].__name__} Score: {score:.4f}\n")
            for score_idx, score in enumerate(tomato_scores):
                f.write(f"Agnostic Tomato (min) {score_list_ranking_1g[score_idx].__name__} Score: {np.min(score):.4f}\n")
                f.write(f"Agnostic Tomato (mean) {score_list_ranking_1g[score_idx].__name__} Score: {np.mean(score):.4f}\n")
            for score_idx, scores in enumerate(baselines_scores):
                for idx, score in enumerate(scores):
                    f.write(f"Baseline {idx+1} {score_list_ranking_1g[score_idx].__name__} Score: {score:.4f}\n")
            
        with open(output_path + dataset + '_' + experiment + '_ranking_times.txt', 'w') as f:
            for idx, tomatomp_time in enumerate(tomatomp_times):
                f.write(f"Tomatomp (nlines = {nlines_list[idx]}) Time: {tomatomp_time:.2f} seconds\n")
            f.write(f"Agnostic Tomato Time: {tomato_time:.2f} seconds\n")
            for idx, baseline_time in enumerate(baseline_times):
                f.write(f"Baseline {idx+1} Time: {baseline_time:.2f} seconds\n")

## Outliers

In [ ]:
if dataset == 'st1gr' and experiment == 'outliers':
    print("outliers")

    num_outliers = int(frac_outliers)

    for seed in seeds:
        
        ## DATASET
    
        np.random.seed(seed)
        dtms_outliers = [(gene, gene_name, dtm.copy()) for (gene, gene_name, dtm) in dtms]
        idx_outliers = np.random.choice(len(dtms[0][2]), size=num_outliers, replace=False)
        for _, _, dtm in dtms_outliers:
            maxval = dtm.max()
            for idx in idx_outliers:
                dtm[idx] = maxval + 100

        ## TOMATO

        start = time.time()
        tomato_ranking = rank_genes_tomato(dtms_outliers, list_neighbors, merge_threshold=None)
        end = time.time()

        tomato_time = end - start

        ## TOMATOMP

        tomatomp_rankings = []
        tomatomp_times = []
        for nlines in nlines_list:
            start = time.time()
            ranking1, ranking2, ranking3, ranking4, ranking5, ranking6 = rank_genes_tomatomp_outlier(dtms_outliers, G, list_neighbors, direction=(1., 1.), nlines=nlines, rescale=rescale)
            tomatomp_rankings.append(ranking1)
            tomatomp_rankings.append(ranking2)
            tomatomp_rankings.append(ranking3)
            tomatomp_rankings.append(ranking4)
            tomatomp_rankings.append(ranking5)
            tomatomp_rankings.append(ranking6)
            end = time.time()
            tomatomp_times.append(end - start)

        ## BASELINES (KMEANS, SPECTRAL CLUSTERING, HIERARCHICAL CLUSTERING, ETC.)

        baseline_rankings = []
        baseline_times = []
        for ranking_methods in baseline_methods_ranking:
            start = time.time()
            ranking = rank_genes_hierarchical(dtms_outliers, spatial_coords, add_outlier_score=False)
            baseline_rankings.append(ranking)
            end = time.time()
            baseline_times.append(end - start)
        for ranking_methods in baseline_methods_ranking:
            start = time.time()
            ranking = rank_genes_hierarchical(dtms_outliers, spatial_coords, add_outlier_score=True, list_neighbors=list_neighbors)
            baseline_rankings.append(ranking)
            end = time.time()
            baseline_times.append(end - start)

        ## SCORES (CORRELATION, MUTUAL INFORMATION, ETC.)

        baselines_scores = [[score(ground_truth_ranking, ranking) for ranking in baseline_rankings] for score in score_list_ranking_1g]
        tomato_scores = [score(ground_truth_ranking, tomato_ranking) for score in score_list_ranking_1g]
        tomatomp_scores = [[score(ground_truth_ranking, ranking) for ranking in tomatomp_rankings] for score in score_list_ranking_1g]

        if mode == 'notebook':
            
            for score_idx, scores in enumerate(tomatomp_scores):
                for idx, score in enumerate(scores):
                    print(f"Tomatomp (nlines = {nlines_list[idx//6]}, mode {idx%6+1}) {score_list_ranking_1g[score_idx].__name__} Score (seed {seed}): {score:.4f}")
            for score_idx, score in enumerate(tomato_scores):
                print(f"Tomato {score_list_ranking_1g[score_idx].__name__} Score (seed {seed}): {score:.4f}")
            for score_idx, scores in enumerate(baselines_scores):
                for idx, score in enumerate(scores):
                    print(f"Baseline {idx+1} {score_list_ranking_1g[score_idx].__name__} Score (seed {seed}): {score:.4f}")
            
            for idx, tomatomp_time in enumerate(tomatomp_times):
                print(f"Tomatomp (nlines = {nlines_list[idx]}) Time (seed {seed}): {tomatomp_time:.2f} seconds")
            print(f"Tomato Time (seed {seed}): {tomato_time:.2f} seconds")
            for idx, baseline_time in enumerate(baseline_times):
                print(f"Baseline {idx+1} Time (seed {seed}): {baseline_time:.2f} seconds")

        else:

            with open(output_path + dataset + '_' + experiment + f'_ranking_scores_seed{seed}.txt', 'w') as f:
                for score_idx, scores in enumerate(tomatomp_scores):
                    for idx, score in enumerate(scores):
                        f.write(f"Tomatomp (nlines = {nlines_list[idx//6]}, mode {idx%6+1}) {score_list_ranking_1g[score_idx].__name__} Score (seed {seed}): {score:.4f}\n")
                for score_idx, score in enumerate(tomato_scores):
                    f.write(f"Tomato {score_list_ranking_1g[score_idx].__name__} Score (seed {seed}): {score:.4f}\n")
                for score_idx, scores in enumerate(baselines_scores):
                    for idx, score in enumerate(scores):
                        f.write(f"Baseline {idx+1} {score_list_ranking_1g[score_idx].__name__} Score (seed {seed}): {score:.4f}\n")
            
            with open(output_path + dataset + '_' + experiment + f'_ranking_times_seed{seed}.txt', 'w') as f:
                for idx, tomatomp_time in enumerate(tomatomp_times):
                    f.write(f"Tomatomp (nlines = {nlines_list[idx]}) Time (seed {seed}): {tomatomp_time:.2f} seconds\n")
                f.write(f"Tomato Time (seed {seed}): {tomato_time:.2f} seconds\n")
                for idx, baseline_time in enumerate(baseline_times):
                    f.write(f"Baseline {idx+1} Time (seed {seed}): {baseline_time:.2f} seconds\n")

# ST 2 Genes Ranking

In [ ]:
if dataset == 'st2gr' and filename.endswith('.csv'):
    
    rescale = True
    
    ## DATASET
    
    df = pd.read_csv(data_path_prefix + '/Data/spatial/persiST/kpmp_30-10125_spatial_expression.csv')
    spatial_coords = np.array(df[['x_position', 'y_position']].values)
    gene_expression = np.array(df.drop(columns=['x_position', 'y_position']).values)
    dtms, st, A, G, list_neighbors = smoothed_expression(df, mesh_type="hexagonal", m=0.1)

In [ ]:
if dataset == 'st2gr' and filename.endswith('.npy'):

    rescale = False

    ## DATASET
    
    spatial_coords = np.load(data_path_prefix + filename)
    dtms = pck.load(open(data_path_prefix + filename.replace('spatial_coords.npy', 'spatial_dtms.pkl'), 'rb'))
    jaccard_ranking = pck.load(open(data_path_prefix + filename.replace('spatial_coords.npy', 'jaccard_results.pkl'), 'rb'))

    spatial_coords = spatial_coords[::subsample]
    dtms = [(idx_g, gene, values[::subsample]) for idx_g, gene, values in dtms]
    A = radius_neighbors_graph(spatial_coords, radius=2.5, mode='connectivity', include_self=False)
    G = nx.from_scipy_sparse_array(A)
    list_neighbors = [list(G.neighbors(i)) for i in range(G.number_of_nodes())]

In [ ]:
if dataset == 'st2gr':

    ## DATASET

    dtm_global_min = min([dtm.min() for (_, _, dtm) in dtms])
    dtm_global_max = max([dtm.max() for (_, _, dtm) in dtms])

    plt.figure()
    plt.scatter(spatial_coords[:, 0], spatial_coords[:, 1], c=dtms[0][2], cmap='viridis', s=10, alpha=1)
    plt.xlabel('X Coordinate')
    plt.ylabel('Y Coordinate')
    plt.grid()
    plt.axis('equal')
    plt.colorbar()
    plt.title('Spatial Coordinates Colored by First Gene Expression')
    if mode == 'notebook':
        plt.show()
    else:
        plt.savefig(output_path + dataset + '_spatial_coordinates.png')

    ## GROUND TRUTH (TOMATO)

    ground_truth_ranking_single = rank_genes_tomato(dtms, list_neighbors, merge_threshold=None)
    top_idxs = np.argsort(ground_truth_ranking_single['score'])[::-1][:top_genes]
    dtms = [dtms[idx] for idx in top_idxs]
    
    ground_truth_ranking = rank_pair_genes_tomato(dtms, list_neighbors, merge_threshold=None)

    if mode == 'notebook':
        print(ground_truth_ranking.sort_values(by='score', ascending=False).head(50))
    else:
        with open(output_path + dataset + '_ground_truth_ranking.txt', 'w') as f:
            for idx, gene_pair in enumerate(ground_truth_ranking.sort_values(by='score', ascending=False)['gene_pair']):
                f.write(f"{idx+1}. {gene_pair}\n")

## No Radius

In [ ]:
if dataset == 'st2gr' and experiment == 'no-radius':
    print("no-radius")

    min_edge_length = quant_min_dist
    max_edge_length = quant_max_dist
    
    ## AGNOSTIC TOMATO

    start = time.time()
    tomato_rankings = []
    for distance_threshold in np.linspace(min_edge_length, max_edge_length, nlines_list[0]):
        A = radius_neighbors_graph(spatial_coords, radius=distance_threshold, mode='connectivity', include_self=False)
        G = nx.from_scipy_sparse_array(A)
        list_neighbors = [list(G.neighbors(i)) for i in range(G.number_of_nodes())]
        tomato_ranking_test = rank_pair_genes_tomato(dtms, list_neighbors, merge_threshold=None)
        tomato_rankings.append(tomato_ranking_test)
    end = time.time()

    tomato_time = end - start

    ## TOMATOMP
    
    tomatomp_rankings = []
    tomatomp_times = []
    for nlines in nlines_list:
        start = time.time()
        ranking1, ranking2, ranking3, ranking4, ranking5, ranking6 = rank_pair_genes_tomatomp_radius(dtms, spatial_coords, direction=(1., 1., 1.), nlines=nlines, min_edge_length=min_edge_length, max_edge_length=max_edge_length, rescale=rescale)
        tomatomp_rankings.append(ranking1)
        tomatomp_rankings.append(ranking2)
        tomatomp_rankings.append(ranking3)
        tomatomp_rankings.append(ranking4)
        tomatomp_rankings.append(ranking5)
        tomatomp_rankings.append(ranking6)
        end = time.time()
        tomatomp_times.append(end - start)

    ## BASELINES (KMEANS, SPECTRAL CLUSTERING, HIERARCHICAL CLUSTERING, ETC.)

    baseline_rankings = []
    baseline_times = []
    for ranking_methods in baseline_methods_ranking:
        start = time.time()
        ranking = rank_pair_genes_hierarchical(dtms, spatial_coords, add_outlier_score=False)
        baseline_rankings.append(ranking)
        end = time.time()
        baseline_times.append(end - start)

    ## SCORES (CORRELATION, MUTUAL INFORMATION, ETC.)

    baselines_scores = [[score(ground_truth_ranking, ranking) for ranking in baseline_rankings] for score in score_list_ranking_2g]
    tomato_scores = [[score(ground_truth_ranking, ranking) for ranking in tomato_rankings] for score in score_list_ranking_2g]
    tomatomp_scores = [[score(ground_truth_ranking, ranking) for ranking in tomatomp_rankings] for score in score_list_ranking_2g]

    if mode == 'notebook':
        
        for score_idx, scores in enumerate(tomatomp_scores):
            for idx, score in enumerate(scores):
                print(f"Tomatomp (nlines = {nlines_list[idx//6]}, mode {idx%6+1}) {score_list_ranking_2g[score_idx].__name__} Score: {score:.4f}")
        for score_idx, score in enumerate(tomato_scores):
            print(f"Agnostic Tomato (min) {score_list_ranking_2g[score_idx].__name__} Score: {np.min(score):.4f}")
            print(f"Agnostic Tomato (mean) {score_list_ranking_2g[score_idx].__name__} Score: {np.mean(score):.4f}")
        for score_idx, scores in enumerate(baselines_scores):
            for idx, score in enumerate(scores):
                print(f"Baseline {idx+1} {score_list_ranking_2g[score_idx].__name__} Score: {score:.4f}")
            
        for idx, tomatomp_time in enumerate(tomatomp_times):
            print(f"Tomatomp (nlines = {nlines_list[idx]}) Time: {tomatomp_time:.2f} seconds")
        print(f"Agnostic Tomato Time: {tomato_time:.2f} seconds")
        for idx, baseline_time in enumerate(baseline_times):
            print(f"Baseline {idx+1} Time: {baseline_time:.2f} seconds")

    else:

        with open(output_path + dataset + '_' + experiment + '_ranking_scores.txt', 'w') as f:
            for score_idx, scores in enumerate(tomatomp_scores):
                for idx, score in enumerate(scores):
                    f.write(f"Tomatomp (nlines = {nlines_list[idx//6]}, mode {idx%6+1}) {score_list_ranking_2g[score_idx].__name__} Score: {score:.4f}\n")
            for score_idx, score in enumerate(tomato_scores):
                f.write(f"Agnostic Tomato (min) {score_list_ranking_2g[score_idx].__name__} Score: {np.min(score):.4f}\n")
                f.write(f"Agnostic Tomato (mean) {score_list_ranking_2g[score_idx].__name__} Score: {np.mean(score):.4f}\n")
            for score_idx, scores in enumerate(baselines_scores):
                for idx, score in enumerate(scores):
                    f.write(f"Baseline {idx+1} {score_list_ranking_2g[score_idx].__name__} Score: {score:.4f}\n")
            
        with open(output_path + dataset + '_' + experiment + '_ranking_times.txt', 'w') as f:
            for idx, tomatomp_time in enumerate(tomatomp_times):
                f.write(f"Tomatomp (nlines = {nlines_list[idx]}) Time: {tomatomp_time:.2f} seconds\n")
            f.write(f"Agnostic Tomato Time: {tomato_time:.2f} seconds\n")
            for idx, baseline_time in enumerate(baseline_times):
                f.write(f"Baseline {idx+1} Time: {baseline_time:.2f} seconds\n")

## Outliers

In [ ]:
if dataset == 'st2gr' and experiment == 'outliers':
    print("outliers")

    num_outliers = int(frac_outliers)

    for seed in seeds:
        
        ## DATASET
    
        np.random.seed(seed)
        dtms_outliers = [(gene, gene_name, dtm.copy()) for (gene, gene_name, dtm) in dtms]
        idx_outliers = np.random.choice(len(dtms[0][2]), size=num_outliers, replace=False)
        for _, _, dtm in dtms_outliers:
            maxval = dtm.max()
            for idx in idx_outliers:
                dtm[idx] = maxval + 100

        ## TOMATO

        start = time.time()
        tomato_ranking = rank_pair_genes_tomato(dtms_outliers, list_neighbors, merge_threshold=None)
        end = time.time()

        tomato_time = end - start

        ## TOMATOMP

        tomatomp_rankings = []
        tomatomp_times = []
        for nlines in nlines_list:
            start = time.time()
            ranking1, ranking2, ranking3, ranking4, ranking5, ranking6 = rank_pair_genes_tomatomp_outlier(dtms_outliers, G, list_neighbors, direction=(1., 1., 1.), nlines=nlines, rescale=rescale)
            tomatomp_rankings.append(ranking1)
            tomatomp_rankings.append(ranking2)
            tomatomp_rankings.append(ranking3)
            tomatomp_rankings.append(ranking4)
            tomatomp_rankings.append(ranking5)
            tomatomp_rankings.append(ranking6)
            end = time.time()
            tomatomp_times.append(end - start)

        ## BASELINES (KMEANS, SPECTRAL CLUSTERING, HIERARCHICAL CLUSTERING, ETC.)

        baseline_rankings = []
        baseline_times = []
        for ranking_methods in baseline_methods_ranking:
            start = time.time()
            ranking = rank_pair_genes_hierarchical(dtms_outliers, spatial_coords, add_outlier_score=False)
            baseline_rankings.append(ranking)
            end = time.time()
            baseline_times.append(end - start)
        for ranking_methods in baseline_methods_ranking:
            start = time.time()
            ranking = rank_pair_genes_hierarchical(dtms_outliers, spatial_coords, add_outlier_score=True, list_neighbors=list_neighbors)
            baseline_rankings.append(ranking)
            end = time.time()
            baseline_times.append(end - start)

        ## SCORES (CORRELATION, MUTUAL INFORMATION, ETC.)

        baselines_scores = [[score(ground_truth_ranking, ranking) for ranking in baseline_rankings] for score in score_list_ranking_2g]
        tomato_scores = [score(ground_truth_ranking, tomato_ranking) for score in score_list_ranking_2g]
        tomatomp_scores = [[score(ground_truth_ranking, ranking) for ranking in tomatomp_rankings] for score in score_list_ranking_2g]

        if mode == 'notebook':
            
            for score_idx, scores in enumerate(tomatomp_scores):
                for idx, score in enumerate(scores):
                    print(f"Tomatomp (nlines = {nlines_list[idx//6]}, mode {idx%6+1}) {score_list_ranking_2g[score_idx].__name__} Score (seed {seed}): {score:.4f}")
            for score_idx, score in enumerate(tomato_scores):
                print(f"Tomato {score_list_ranking_2g[score_idx].__name__} Score (seed {seed}): {score:.4f}")
            for score_idx, scores in enumerate(baselines_scores):
                for idx, score in enumerate(scores):
                    print(f"Baseline {idx+1} {score_list_ranking_2g[score_idx].__name__} Score (seed {seed}): {score:.4f}")
            
            for idx, tomatomp_time in enumerate(tomatomp_times):
                print(f"Tomatomp (nlines = {nlines_list[idx]}) Time (seed {seed}): {tomatomp_time:.2f} seconds")
            print(f"Tomato Time (seed {seed}): {tomato_time:.2f} seconds")
            for idx, baseline_time in enumerate(baseline_times):
                print(f"Baseline {idx+1} Time (seed {seed}): {baseline_time:.2f} seconds")

        else:

            with open(output_path + dataset + '_' + experiment + f'_ranking_scores_seed{seed}.txt', 'w') as f:
                for score_idx, scores in enumerate(tomatomp_scores):
                    for idx, score in enumerate(scores):
                        f.write(f"Tomatomp (nlines = {nlines_list[idx//6]}, mode {idx%6+1}) {score_list_ranking_2g[score_idx].__name__} Score (seed {seed}): {score:.4f}\n")
                for score_idx, score in enumerate(tomato_scores):
                    f.write(f"Tomato {score_list_ranking_2g[score_idx].__name__} Score (seed {seed}): {score:.4f}\n")
                for score_idx, scores in enumerate(baselines_scores):
                    for idx, score in enumerate(scores):
                        f.write(f"Baseline {idx+1} {score_list_ranking_2g[score_idx].__name__} Score (seed {seed}): {score:.4f}\n")
            
            with open(output_path + dataset + '_' + experiment + f'_ranking_times_seed{seed}.txt', 'w') as f:
                for idx, tomatomp_time in enumerate(tomatomp_times):
                    f.write(f"Tomatomp (nlines = {nlines_list[idx]}) Time (seed {seed}): {tomatomp_time:.2f} seconds\n")
                f.write(f"Tomato Time (seed {seed}): {tomato_time:.2f} seconds\n")
                for idx, baseline_time in enumerate(baseline_times):
                    f.write(f"Baseline {idx+1} Time (seed {seed}): {baseline_time:.2f} seconds\n")

## Dimension

In [ ]:
if dataset == 'st2gr' and experiment == 'dimension':

    scores_mod1 = []
    scores_mod2 = []
    scores_mod3 = []
    scores_mod4 = []
    scores_mod5 = []
    scores_mod6 = []
    for idx1, (_,gene1,dtm1) in enumerate(dtms):
        for idx2, (_,gene2,dtm2) in enumerate(dtms[idx1+1:]):
            for _, (_,gene3,dtm3) in enumerate(dtms[idx2+idx1+2:]):

                start = time.time()
                tomatomp = Tomatomp(
                    direction=(1., 1., 1.),
                    slice_number=nlines_list[0],
                    bounding_box=np.array([[np.nan, np.nan, np.nan], [np.nan, np.nan, np.nan]]),
                    merging_threshold=None,
                    n_clusters=None,
                    sigma2=0.,
                    rescale=False,
                    #scale_filts=[1., 1., 1.],
                    verbose=False,
                )
                tomatomp.fit(G, weights=np.hstack([-dtm1[:,None], -dtm2[:,None], -dtm3[:,None]]))
                end = time.time()

                scores_mod1.append(( (gene1, gene2, gene3), mma_score(tomatomp.mma, tomatomp.bounding_box[0,:], tomatomp.bounding_box[1,:], tomatomp.direction, tomatomp.slice_number, mode=1, quant=0.1, plot_dgms=False)))
                scores_mod2.append(( (gene1, gene2, gene3), mma_score(tomatomp.mma, tomatomp.bounding_box[0,:], tomatomp.bounding_box[1,:], tomatomp.direction, tomatomp.slice_number, mode=1, quant=0.5, plot_dgms=False)))
                scores_mod3.append(( (gene1, gene2, gene3), mma_score(tomatomp.mma, tomatomp.bounding_box[0,:], tomatomp.bounding_box[1,:], tomatomp.direction, tomatomp.slice_number, mode=1, quant=0.9, plot_dgms=False)))
                scores_mod4.append(( (gene1, gene2, gene3), mma_score(tomatomp.mma, tomatomp.bounding_box[0,:], tomatomp.bounding_box[1,:], tomatomp.direction, tomatomp.slice_number, mode=2, quant=0.1, plot_dgms=False)))
                scores_mod5.append(( (gene1, gene2, gene3), mma_score(tomatomp.mma, tomatomp.bounding_box[0,:], tomatomp.bounding_box[1,:], tomatomp.direction, tomatomp.slice_number, mode=2, quant=0.5, plot_dgms=False)))
                scores_mod6.append(( (gene1, gene2, gene3), mma_score(tomatomp.mma, tomatomp.bounding_box[0,:], tomatomp.bounding_box[1,:], tomatomp.direction, tomatomp.slice_number, mode=2, quant=0.9, plot_dgms=False)))

    scores_mod1.sort(key=lambda x: x[1], reverse=True)
    scores_mod2.sort(key=lambda x: x[1], reverse=True)
    scores_mod3.sort(key=lambda x: x[1], reverse=True)
    scores_mod4.sort(key=lambda x: x[1], reverse=True)
    scores_mod5.sort(key=lambda x: x[1], reverse=True)
    scores_mod6.sort(key=lambda x: x[1], reverse=True)

    if mode == 'notebook':
        print(f"Time taken for computing MMA scores for all gene triples: {end - start:.2f} seconds")
        print("Top gene triples by MMA Score (mode 1, quant 0.1):")
        for (gene1, gene2, gene3), score in scores_mod1[:10]:
            print(f"{gene1} - {gene2} - {gene3}: {score:.4f}")
        print("Top gene triples by MMA Score (mode 1, quant 0.5):")
        for (gene1, gene2, gene3), score in scores_mod2[:10]:
            print(f"{gene1} - {gene2} - {gene3}: {score:.4f}")
        print("Top gene triples by MMA Score (mode 1, quant 0.9):")
        for (gene1, gene2, gene3), score in scores_mod3[:10]:
            print(f"{gene1} - {gene2} - {gene3}: {score:.4f}")
        print("Top gene triples by MMA Score (mode 2, quant 0.1):")
        for (gene1, gene2, gene3), score in scores_mod4[:10]:
            print(f"{gene1} - {gene2} - {gene3}: {score:.4f}")
        print("Top gene triples by MMA Score (mode 2, quant 0.5):")
        for (gene1, gene2, gene3), score in scores_mod5[:10]:
            print(f"{gene1} - {gene2} - {gene3}: {score:.4f}")
        print("Top gene triples by MMA Score (mode 2, quant 0.9):")
        for (gene1, gene2, gene3), score in scores_mod6[:10 ]:
            print(f"{gene1} - {gene2} - {gene3}: {score:.4f}")
    else:
        with open(output_path + dataset + '_' + experiment + '_top_gene_triples.txt', 'w') as f:
            f.write(f"Time taken for computing MMA scores for all gene triples: {end - start:.2f} seconds\n")
            f.write("Top gene triples by MMA Score (mode 1, quant 0.1):\n")
            for (gene1, gene2, gene3), score in scores_mod1[:10]:
                f.write(f"{gene1} - {gene2} - {gene3}: {score:.4f}\n")
            f.write("Top gene triples by MMA Score (mode 1, quant 0.5):\n")
            for (gene1, gene2, gene3), score in scores_mod2[:10]:
                f.write(f"{gene1} - {gene2} - {gene3}: {score:.4f}\n")
            f.write("Top gene triples by MMA Score (mode 1, quant 0.9):\n")
            for (gene1, gene2, gene3), score in scores_mod3[:10]:
                f.write(f"{gene1} - {gene2} - {gene3}: {score:.4f}\n")
            f.write("Top gene triples by MMA Score (mode 2, quant 0.1):\n")
            for (gene1, gene2, gene3), score in scores_mod4[:10]:
                f.write(f"{gene1} - {gene2} - {gene3}: {score:.4f}\n")
            f.write("Top gene triples by MMA Score (mode 2, quant 0.5):\n")
            for (gene1, gene2, gene3), score in scores_mod5[:10]:
                f.write(f"{gene1} - {gene2} - {gene3}: {score:.4f}\n")
            f.write("Top gene triples by MMA Score (mode 2, quant 0.9):\n")
            for (gene1, gene2, gene3), score in scores_mod6[:10]:
                f.write(f"{gene1} - {gene2} - {gene3}: {score:.4f}\n")

    most_frequent_triples = {}
    for scores in [scores_mod1, scores_mod2, scores_mod3, scores_mod4, scores_mod5, scores_mod6]:
        for (gene1, gene2, gene3), score in scores[:10]:
            triple = tuple(sorted([gene1, gene2, gene3]))
            if triple not in most_frequent_triples:
                most_frequent_triples[triple] = 0
            most_frequent_triples[triple] += 1

    most_frequent_triples = sorted(most_frequent_triples.items(), key=lambda x: x[1], reverse=True)

    if mode == 'notebook':
        print("Most frequent gene triples across top 10 of all scoring methods:")
        for (gene1, gene2, gene3), count in most_frequent_triples[:10]:
            print(f"{gene1} - {gene2} - {gene3}: {count} times")

    else:

        with open(output_path + dataset + '_' + experiment + '_most_frequent_triples.txt', 'w') as f:
            f.write("Most frequent gene triples across top 10 of all scoring methods:\n")
            for (gene1, gene2, gene3), count in most_frequent_triples[:10]:
                f.write(f"{gene1} - {gene2} - {gene3}: {count} times\n")

    for gene_triple, count in most_frequent_triples[:10]:
        gene1, gene2, gene3 = gene_triple
        plt.figure(figsize=(15, 5))
        for idx, gene in enumerate([gene1, gene2, gene3]):
            dtm = next(dtm for (g, gene_name, dtm) in dtms if gene_name == gene)
            plt.subplot(1, 3, idx+1)
            plt.scatter(spatial_coords[:, 0], spatial_coords[:, 1], c=dtm, cmap='viridis', s=100*dtm, alpha=1)
            plt.xlabel('X Coordinate')
            plt.ylabel('Y Coordinate')
            plt.grid()
            plt.colorbar()
            plt.title(f'Spatial Expression of {gene}')
        plt.suptitle(f'Spatial Expression of Triple: {gene1} - {gene2} - {gene3} (top of ranking)')
        plt.tight_layout()
        if mode == 'notebook':
            plt.show()
        else:
            plt.savefig(output_path + dataset + '_' + experiment + f'_{gene1}_{gene2}_{gene3}_spatial_expression_top.png')

    bad_triples = [triple for triple,_ in scores_mod1[-5:-1]]
    for idx, triple in enumerate(bad_triples):
        plt.figure(figsize=(15, 5))
        for jdx, gene in enumerate(triple):
            dtm = next(dtm for (g, gene_name, dtm) in dtms if gene_name == gene)
            plt.subplot(1, 3, jdx+1)
            plt.scatter(spatial_coords[:, 0], spatial_coords[:, 1], c=dtm, cmap='viridis', s=100*dtm, alpha=1)
            plt.xlabel('X Coordinate')
            plt.ylabel('Y Coordinate')
            plt.grid()
            plt.colorbar()
            plt.title(f'Spatial Expression of {gene}')
        plt.suptitle(f'Spatial Expression of Triple: {triple[0]} - {triple[1]} - {triple[2]} (bottom of ranking)')
        plt.tight_layout()
    if mode == 'notebook':
        plt.show()
    else:
        plt.savefig(output_path + dataset + '_' + experiment + f'_{triple[0]}_{triple[1]}_{triple[2]}_spatial_expression_bottom.png')